In [ ]:
#Downloading the dataset
import kagglehub

path = kagglehub.dataset_download("arseniyshutko/binary-aviation-satisfaction-129k", )
print("Path to dataset files:", path)

In [2]:
#Summary of the Dataset
import os
import pandas as pd

csv_file = os.path.join(path, "data.csv")   # adjust filename if different
data = pd.read_csv(csv_file)
print(data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129880 entries, 0 to 129879
Data columns (total 22 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   Gender                             129880 non-null  object 
 1   Customer Type                      129880 non-null  object 
 2   Age                                129880 non-null  int64  
 3   Type of Travel                     129880 non-null  object 
 4   Class                              129880 non-null  object 
 5   Flight Distance                    129880 non-null  int64  
 6   Inflight wifi service              129880 non-null  int64  
 7   Departure/Arrival time convenient  129880 non-null  int64  
 8   Ease of Online booking             129880 non-null  int64  
 9   Gate location                      129880 non-null  int64  
 10  Food and drink                     129880 non-null  int64  
 11  Online boarding                    1298

In [4]:
#Importing the Classification Models and Evaluation Metrics 
import numpy as np
import os

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    LabelEncoder
)

from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    matthews_corrcoef
)
os.makedirs("models", exist_ok=True)
print("Classification Models and Evaluation Metrics Imported!")

Classification Models and Evaluation Metrics Imported!


In [5]:
#Pre-processing the input data
import joblib

data.dropna(inplace=True) #Handle missing values

# Drop ID column if present
for col in ["ID", "id", "Passenger_ID"]:
    if col in data.columns:
        data.drop(columns=[col], inplace=True)

TARGET = "satisfaction"

X = data.drop(TARGET, axis=1)
y = data[TARGET]

#Encode Categorical variables
le = LabelEncoder()
y = le.fit_transform(y)

joblib.dump(le, "models/target_encoder.pkl")

#Feature Types
categorical_cols = X.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_cols = X.select_dtypes(
    exclude=["object"]
).columns.tolist()

#Preprocessor
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numerical_cols),
    ("cat", categorical_transformer, categorical_cols)
])

In [6]:
#Train-Test-Split Dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

test_df = X_test.copy()
test_df[TARGET] = y_test

test_df.to_csv(
    "test_data.csv",
    index=False
)

In [7]:
#Classification Models
models = {
    "Logistic Regression":
        LogisticRegression(max_iter=2000),

    "Decision Tree":
        DecisionTreeClassifier(

            max_depth=10,
            random_state=42
        ),

    "KNN":
        KNeighborsClassifier(n_neighbors=5),

    "SVM":
        SVC(
            probability=True,
            random_state=42
        )
}

In [8]:
#Evaluation Metrics
results = []

for name, clf in models.items():

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", clf)
    ])

    pipe.fit(X_train, y_train)

    preds = pipe.predict(X_test)

    probs = pipe.predict_proba(X_test)[:,1]

    metrics = {
        "Model": name,
        "Accuracy":
            accuracy_score(y_test,preds),

        "AUC":
            roc_auc_score(y_test,probs),

        "Precision":
            precision_score(y_test,preds),

        "Recall":
            recall_score(y_test,preds),

        "F1":
            f1_score(y_test,preds),

        "MCC":
            matthews_corrcoef(y_test,preds)
    }

    results.append(metrics)

    filename = (
        name.lower()
        .replace(" ","_")
        + ".pkl"
    )

    joblib.dump(
        pipe,
        f"models/{filename}"
    )

/home/s2026aj05445/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [9]:
print(results)

[{'Model': 'Logistic Regression', 'Accuracy': 0.8738126496254537, 'AUC': 0.9269070420036241, 'Precision': 0.8673291011132579, 'Recall': 0.8377321603128055, 'F1': 0.8522737546333966, 'MCC': 0.7425423437999803}, {'Model': 'Decision Tree', 'Accuracy': 0.9447061549154375, 'AUC': 0.9863458284345978, 'Precision': 0.9520390315750713, 'Recall': 0.9190438105394118, 'F1': 0.9352504973774642, 'MCC': 0.8874488378841241}, {'Model': 'KNN', 'Accuracy': 0.9293381728318789, 'AUC': 0.9714268737016285, 'Precision': 0.9439366814284368, 'Recall': 0.8902514884919577, 'F1': 0.9163084240373182, 'MCC': 0.856354238289842}, {'Model': 'SVM', 'Accuracy': 0.9530851803228049, 'AUC': 0.9886039144919649, 'Precision': 0.9570205791294846, 'Recall': 0.9339731627121657, 'F1': 0.9453564200584664, 'MCC': 0.9044718880344019}]
